In [1]:

# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
import os
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
# %% [markdown]
## Configure Settings
# Set options or configurations as needed

# Example: pd.set_option('display.max_columns', None)


# Train Data

In [4]:
dftrain = pd.read_csv(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Model_Data_Set_preparation\Tendo_Data_Preparation\Credo_Reference\tendo_data_credo_train_data_14012026.csv", low_memory = False)

# Test Data

In [3]:
dftest = pd.read_csv(r'D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Model_Data_Set_preparation\Tendo_Data_Preparation\Credo_Reference\tendo_data_credo_test_data_14012026.csv', low_memory = False)

In [5]:
print(f"The shape of the train dataset is:\t {dftrain.shape}")
print(f"The shape of the test dataset is:\t {dftest.shape}")

The shape of the train dataset is:	 (10703, 3)
The shape of the test dataset is:	 (2511, 3)


In [6]:
dftrain.columns

Index(['ee_customer_id', 'last_tpsd_date', 'ee_resignation_date_correct'], dtype='object')

In [9]:
dftrain.sample(10)

,ee_customer_id,last_tpsd_date,ee_resignation_date_correct
8365,1295266,2025-04-30,2025-06-16 00:00:00
9508,1321811,2025-03-07,2025-04-03 00:00:00
6972,1262004,2025-01-27,2025-02-18 00:00:00
9724,1332653,2025-05-16,2025-06-04 00:00:00
6622,1250156,2025-01-21,2025-05-29 00:00:00
10084,363941,2025-01-31,2025-01-21 00:00:00
10153,510342,2025-01-15,2025-01-23 00:00:00
1088,1131136,2025-03-17,2025-04-14 00:00:00
1960,1151590,2025-02-28,2025-03-19 00:00:00
4695,1211039,2025-03-04,2025-03-25 00:00:00


In [7]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.temp_find_credoreference_train_data"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dftrain, table_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=bd4fe7f7-282e-4b6d-b894-2e8110f8e34e>

In [8]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.temp_find_credoreference_test_data"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dftest, table_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=1273c15a-419c-487d-b597-553d1e2472f7>

In [10]:
sq = """select td.ee_customer_id, date(td.last_tpsd_date) last_tpsd_date,
date(td.ee_resignation_date_correct) ee_resignation_date_correct,
user_timelines.first_account_activated_at as ee_onboarding_date
,case when date(clrn.created_at) between date(user_timelines.first_account_activated_at) and date(td.last_tpsd_date) then clrn.reference_number end credo_reference_number1
,case when date(clrn.created_at)between date(user_timelines.first_account_activated_at) and date(td.last_tpsd_date) then date(clrn.created_at) end credo_inquiry_date_1
,case when date(clrn.created_at) between date(user_timelines.first_account_activated_at) and date(td.ee_resignation_date_correct) then clrn.reference_number end credo_reference_number2
,case when date(clrn.created_at)between date(user_timelines.first_account_activated_at) and date(td.ee_resignation_date_correct) then date(clrn.created_at) end credo_inquiry_date_2
from prj-prod-dataplatform.dap_ds_poweruser_playground.temp_find_credoreference_train_data td
left join 
(select * from prj-prod-dataplatform.tendopay_raw.credo_lab_reference_numbers_migrated
union all 
select * from prj-prod-dataplatform.tendopay_raw.credo_lab_reference_numbers
)
 clrn on clrn.user_id = td.ee_customer_id
LEFT JOIN  tendopay_raw.user_timelines  on user_timelines.user_id = td.ee_customer_id
order by 1,2,3;"""

df_train_result = client.query(sq).to_dataframe()
df_train_result.to_csv(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Model_Data_Set_preparation\Tendo_Data_Preparation\Credo_Reference\tendo_reference_train_dataset.csv", index = False)

In [11]:
sq = """select td.ee_customer_id, date(td.last_tpsd_date) last_tpsd_date,
date(td.ee_resignation_date_correct) ee_resignation_date_correct,
user_timelines.first_account_activated_at as ee_onboarding_date
,case when date(clrn.created_at) between date(user_timelines.first_account_activated_at) and date(td.last_tpsd_date) then clrn.reference_number end credo_reference_number1
,case when date(clrn.created_at)between date(user_timelines.first_account_activated_at) and date(td.last_tpsd_date) then date(clrn.created_at) end credo_inquiry_date_1
,case when date(clrn.created_at) between date(user_timelines.first_account_activated_at) and date(td.ee_resignation_date_correct) then clrn.reference_number end credo_reference_number2
,case when date(clrn.created_at)between date(user_timelines.first_account_activated_at) and date(td.ee_resignation_date_correct) then date(clrn.created_at) end credo_inquiry_date_2
from prj-prod-dataplatform.dap_ds_poweruser_playground.temp_find_credoreference_test_data td
left join 
(select * from prj-prod-dataplatform.tendopay_raw.credo_lab_reference_numbers_migrated
union all 
select * from prj-prod-dataplatform.tendopay_raw.credo_lab_reference_numbers
)
 clrn on clrn.user_id = td.ee_customer_id
LEFT JOIN  tendopay_raw.user_timelines  on user_timelines.user_id = td.ee_customer_id
order by 1,2,3;"""

df_train_result = client.query(sq).to_dataframe()
df_train_result.to_csv(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Model_Data_Set_preparation\Tendo_Data_Preparation\Credo_Reference\tendo_reference_test_dataset.csv", index = False)